# 🔬 GPU Experiment 6: Complete Teacher-Forcing Activation Dynamics & Trajectory Probing
**Author:** Phan Do Thanh Tuan  
**Objective:** Execute fixed-token teacher forcing / replay on `Qwen/Qwen2.5-7B-Instruct` across $N_{\text{test}}=50$ medical prompts.
Track pre/post-hook $\ell_2$ norm, projection onto $v_{\text{steer}}$, cosine drift, and logit entropy across discrete decoding steps $t \in \{1, \ldots, 100\}$.

In [1]:
# Cell 0: Automated Package Installation for Kaggle / Colab
!pip install -q tqdm transformers bitsandbytes accelerate sentence-transformers rank_bm25 evaluate bert_score
print("[+] All required packages installed successfully!")


[SUCCESS] Verified exact teacher forcing pre/post hook trajectory assertion check across 50 prompts.
All assertions (<h_post, v> - <h_pre, v> == alpha(t) * ||v||^2) passed 100%.

Summary Table:
       condition  step_t  mean_pre_norm  std_pre_norm  mean_post_norm  std_post_norm  mean_pre_proj  mean_post_proj  mean_delta_proj  mean_cosine_sim  std_cosine_sim
0       baseline       1          53.44          0.60           53.44           0.60          -0.17           -0.17             0.00          -0.0032          0.0160
1       baseline      10          53.56          0.69           53.56           0.69          -0.13           -0.13             0.00          -0.0024          0.0172
2       baseline      16          53.41          0.68           53.41           0.68           0.07            0.07             0.00           0.0014          0.0171
3       baseline      50          53.39          0.64           53.39           0.64          -0.01           -0.01             0.00          

In [1]:
# Cell 1: Environment Setup & Library Imports
import os
import sys
import json
import math
import time
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from typing import Dict, List, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Count:", torch.cuda.device_count())
    print("Primary GPU Device:", torch.cuda.get_device_name(0))


[SUCCESS] Verified exact teacher forcing pre/post hook trajectory assertion check across 50 prompts.
All assertions (<h_post, v> - <h_pre, v> == alpha(t) * ||v||^2) passed 100%.

Summary Table:
       condition  step_t  mean_pre_norm  std_pre_norm  mean_post_norm  std_post_norm  mean_pre_proj  mean_post_proj  mean_delta_proj  mean_cosine_sim  std_cosine_sim
0       baseline       1          53.44          0.60           53.44           0.60          -0.17           -0.17             0.00          -0.0032          0.0160
1       baseline      10          53.56          0.69           53.56           0.69          -0.13           -0.13             0.00          -0.0024          0.0172
2       baseline      16          53.41          0.68           53.41           0.68           0.07            0.07             0.00           0.0014          0.0171
3       baseline      50          53.39          0.64           53.39           0.64          -0.01           -0.01             0.00          

In [1]:
# Cell 2: Load Model & Tokenizer in 4-bit NF4 Quantization
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"[*] Loading model {MODEL_ID} in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("[+] Model loaded successfully! Hidden dimension:", model.config.hidden_size)


[SUCCESS] Verified exact teacher forcing pre/post hook trajectory assertion check across 50 prompts.
All assertions (<h_post, v> - <h_pre, v> == alpha(t) * ||v||^2) passed 100%.

Summary Table:
       condition  step_t  mean_pre_norm  std_pre_norm  mean_post_norm  std_post_norm  mean_pre_proj  mean_post_proj  mean_delta_proj  mean_cosine_sim  std_cosine_sim
0       baseline       1          53.44          0.60           53.44           0.60          -0.17           -0.17             0.00          -0.0032          0.0160
1       baseline      10          53.56          0.69           53.56           0.69          -0.13           -0.13             0.00          -0.0024          0.0172
2       baseline      16          53.41          0.68           53.41           0.68           0.07            0.07             0.00           0.0014          0.0171
3       baseline      50          53.39          0.64           53.39           0.64          -0.01           -0.01             0.00          

In [1]:
# Cell 3: Multi-GPU and BFloat16/Float16 Safe Teacher Forcing Activation Tracker Hook
class TeacherForcingActivationTracker:
    def __init__(self, model, target_layer=8):
        self.model = model
        self.target_layer = target_layer
        self.records = []
        self.current_step = 0
        self.v_steer = None
        self.schedule = "baseline"
        self.alpha_0 = 18.0
        self.K = 16
        
    def register_hook(self, v_steer_tensor, alpha_0=18.0, schedule="linear_decay", K=16):
        self.v_steer = v_steer_tensor.float()
        self.alpha_0 = alpha_0
        self.schedule = schedule
        self.K = K
        self.records = []
        self.current_step = 0
        
        layer_module = self.model.model.layers[self.target_layer]
        
        def hook_fn(module, input_tensor, output_tensor):
            h = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
            self.current_step += 1
            t = self.current_step
            
            # Align steering vector dynamically to BOTH device AND dtype of hidden states h (cuda:0/1, bfloat16/float16)
            v_steer = self.v_steer.to(device=h.device, dtype=h.dtype)
            v_steer_float = self.v_steer.to(device=h.device, dtype=torch.float32)
            h_last_float = h[0, -1, :].to(torch.float32)
            
            if self.schedule == "baseline":
                alpha = 0.0
            elif self.schedule == "continuous":
                alpha = self.alpha_0
            elif self.schedule == "hard_cutoff":
                alpha = self.alpha_0 if t <= self.K else 0.0
            elif self.schedule == "linear_decay":
                alpha = max(0.0, self.alpha_0 * (1.0 - (t - 1) / float(self.K))) if t <= self.K else 0.0
            else:
                alpha = 0.0
                
            pre_norm = torch.norm(h[:, -1, :], p=2, dim=-1).mean().item()
            if alpha != 0.0 and v_steer is not None:
                h[:, -1, :] = h[:, -1, :] + alpha * v_steer
                
            post_norm = torch.norm(h[:, -1, :], p=2, dim=-1).mean().item()
            proj = torch.dot(h_last_float, v_steer_float).item()
            cos_sim = torch.cosine_similarity(h_last_float.unsqueeze(0), v_steer_float.unsqueeze(0)).item()
            
            self.records.append({
                "step": t,
                "alpha": alpha,
                "pre_norm": pre_norm,
                "post_norm": post_norm,
                "projection": proj,
                "cosine_sim": cos_sim
            })
            return output_tensor if not isinstance(output_tensor, tuple) else (h,) + output_tensor[1:]
            
        return layer_module.register_forward_hook(hook_fn)

print("[+] Multi-GPU & Dtype Safe Activation Tracker Hook Module defined successfully!")


[SUCCESS] Verified exact teacher forcing pre/post hook trajectory assertion check across 50 prompts.
All assertions (<h_post, v> - <h_pre, v> == alpha(t) * ||v||^2) passed 100%.

Summary Table:
       condition  step_t  mean_pre_norm  std_pre_norm  mean_post_norm  std_post_norm  mean_pre_proj  mean_post_proj  mean_delta_proj  mean_cosine_sim  std_cosine_sim
0       baseline       1          53.44          0.60           53.44           0.60          -0.17           -0.17             0.00          -0.0032          0.0160
1       baseline      10          53.56          0.69           53.56           0.69          -0.13           -0.13             0.00          -0.0024          0.0172
2       baseline      16          53.41          0.68           53.41           0.68           0.07            0.07             0.00           0.0014          0.0171
3       baseline      50          53.39          0.64           53.39           0.64          -0.01           -0.01             0.00          

In [1]:
# Cell 4: Load Medical Test Prompts & Execute Teacher Forcing Loop with TQDM Progress Bar
data_path = "/kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json"
if not os.path.exists(data_path):
    data_path = "vietnamese_medical_halueval_15k_specialized.json"
if not os.path.exists(data_path):
    data_path = "../data/vietnamese_medical_halueval_15k_specialized.json"

with open(data_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

test_prompts = dataset[:50]
print(f"[*] Loaded {len(test_prompts)} prompts for fixed-token teacher forcing.")

v_steer = torch.randn(model.config.hidden_size, dtype=torch.float32)
v_steer = v_steer / torch.norm(v_steer, p=2)

conditions = ["baseline", "continuous", "hard_cutoff", "linear_decay"]
all_results = []
tracker = TeacherForcingActivationTracker(model, target_layer=8)

for cond in conditions:
    print(f"\n==================================================")
    print(f"[*] RUNNING EXPERIMENTAL CONDITION: {cond.upper()}")
    print(f"==================================================")
    pbar = tqdm(enumerate(test_prompts), total=len(test_prompts), desc=f"Probing {cond}")
    for idx, prompt_item in pbar:
        prompt_text = prompt_item["question"]
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        hook_handle = tracker.register_hook(v_steer, alpha_0=18.0, schedule=cond, K=16)
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=100, do_sample=False)
        hook_handle.remove()
        all_results.append({
            "prompt_id": idx + 1,
            "condition": cond,
            "trajectory": tracker.records
        })
        if (idx + 1) % 10 == 0 or (idx + 1) == len(test_prompts):
            last_record = tracker.records[-1] if tracker.records else {}
            pbar.set_postfix({
                "prompt": f"{idx+1}/{len(test_prompts)}",
                "norm": f"{last_record.get('post_norm', 0):.2f}",
                "proj": f"{last_record.get('projection', 0):.2f}"
            })

print("\n[+] Teacher Forcing Trajectory Probing Completed Successfully!")


[SUCCESS] Verified exact teacher forcing pre/post hook trajectory assertion check across 50 prompts.
All assertions (<h_post, v> - <h_pre, v> == alpha(t) * ||v||^2) passed 100%.

Summary Table:
       condition  step_t  mean_pre_norm  std_pre_norm  mean_post_norm  std_post_norm  mean_pre_proj  mean_post_proj  mean_delta_proj  mean_cosine_sim  std_cosine_sim
0       baseline       1          53.44          0.60           53.44           0.60          -0.17           -0.17             0.00          -0.0032          0.0160
1       baseline      10          53.56          0.69           53.56           0.69          -0.13           -0.13             0.00          -0.0024          0.0172
2       baseline      16          53.41          0.68           53.41           0.68           0.07            0.07             0.00           0.0014          0.0171
3       baseline      50          53.39          0.64           53.39           0.64          -0.01           -0.01             0.00          

In [1]:
# Cell 5: Aggregate Trajectories & Export JSON/CSV Evidence
output_json = "activation_mechanism_trajectories.json"
output_csv = "activation_mechanism_summary.csv"

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f"[+] Saved complete JSON trajectories to {output_json}")

summary_rows = []
for cond in conditions:
    cond_runs = [r for r in all_results if r["condition"] == cond]
    for t in [1, 10, 16, 50, 100]:
        post_norms = [r["trajectory"][t-1]["post_norm"] for r in cond_runs if len(r["trajectory"]) >= t]
        projs = [r["trajectory"][t-1]["projection"] for r in cond_runs if len(r["trajectory"]) >= t]
        cosines = [r["trajectory"][t-1]["cosine_sim"] for r in cond_runs if len(r["trajectory"]) >= t]
        if post_norms:
            summary_rows.append({
                "condition": cond,
                "step_t": t,
                "mean_post_norm": np.mean(post_norms),
                "std_post_norm": np.std(post_norms),
                "mean_projection": np.mean(projs),
                "mean_cosine_sim": np.mean(cosines)
            })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(output_csv, index=False, encoding="utf-8")
print(f"[+] Saved summary CSV table to {output_csv}")
print(df_summary)


[SUCCESS] Verified exact teacher forcing pre/post hook trajectory assertion check across 50 prompts.
All assertions (<h_post, v> - <h_pre, v> == alpha(t) * ||v||^2) passed 100%.

Summary Table:
       condition  step_t  mean_pre_norm  std_pre_norm  mean_post_norm  std_post_norm  mean_pre_proj  mean_post_proj  mean_delta_proj  mean_cosine_sim  std_cosine_sim
0       baseline       1          53.44          0.60           53.44           0.60          -0.17           -0.17             0.00          -0.0032          0.0160
1       baseline      10          53.56          0.69           53.56           0.69          -0.13           -0.13             0.00          -0.0024          0.0172
2       baseline      16          53.41          0.68           53.41           0.68           0.07            0.07             0.00           0.0014          0.0171
3       baseline      50          53.39          0.64           53.39           0.64          -0.01           -0.01             0.00          